In [ ]:
# Paths are resolved relative to the repository, or to $PUBO_DATA if set.
# The original research code used absolute paths on the authors' machine;
# only the root has been parameterised, the directory layout is unchanged.
import os
DATA_ROOT = os.environ.get('PUBO_DATA',
                           os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'legacy'))
EXTERNAL_ROOT = os.environ.get('PUBO_EXTERNAL', DATA_ROOT)  # data not shipped with the repo


In [ ]:
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import time
import pandas as p
from scipy import pi
import random
from IPython.display import clear_output

import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv

In [ ]:
 # amplitude increment
class SIMCIM():
    #Initial parameters
    def __init__(self,  J, alpha,rho,delta,gamma, pumping, N,dt, dim,  zeta, attempt_num,  device='cuda'):
        #PUBO matrices and offset
        self.J=torch.Tensor.float(J).to(device)

        #self.lastcuts = 0
        #Parameters of the algorithm (for adjusting)
        #self.const=const
        self.N=N
        self.alpha=alpha
        self.rho=rho
        self.delta=delta
        self.gamma=gamma
        self.zeta=zeta
        self.dt=dt
        self.pumping=pumping
        #Parameters of the algorithm (will not be changed)
        self.attempt_num = attempt_num
        self.device = device
        self.dim=self.J.size(0)
        self.tau=3*self.dim
        self.norm_zeta=torch.mean(torch.sum(torch.sum(torch.abs(J), axis=0), axis=0)).item()/3
        self.size = self.J.size(0)
        self.c_str = 1 /(torch.sum(torch.abs(J))/self.dim/self.dim/self.dim)
        self.GGamma = 100.
#         self.delta1 = self.delta / self.size
#         self.zeta1 = self.zeta / self.norm_zeta
#         self.gamma1 = self.gamma / self.size
        #self.lastcuts=torch.zeros(batch)
    #amplitude increment
   
    # pump ramp
    
#     def pump(self, pumping, rho, C_opt, Cut, delta):
#         return (pumping + rho*torch.tanh(-delta*(C_opt-Cut)))

#     def calc_energy(self, c):
#         sign=torch.sign(c).cuda('cuda')
#         sign[sign==0] = 1.0
#         energy=-0.5*torch.einsum('ij,ik,jk->k',(J,sign,sign)) 
#         return energy
    def ampl_inc(self,p, c, er):
        return (-1+p)*c - c*c*c - self.zeta/self.norm_zeta*er*torch.einsum('ijk,js,ks->is',(self.J, c, c))

    def error_inc(self, c, at, er, betha):
        return -betha*(c*c-at)*er
 
  

    def tanh(self, C_opt, Cut):
        return self.rho * torch.tanh(self.delta * (-C_opt+Cut))
    
#     def ta(self, C_opt, Cut):
#         return self.alpha - self.rho*torch.tanh(self.delta1 * (-C_opt+Cut))
#     def pump(self, C_opt, Cut):
#         return self.pumping + self.rho*torch.tanh(self.delta1 * (-C_opt+Cut))

    def beta (self, t1):
        return t1*self.gamma / self.size
    
    # evolution of amplitudes
    # init_value -- init coupling
    # final_value -- final coupling
    # N -- number of time iterations
    # attempt_num -- number of runs 
    # J -- coupling matrix
    # b -- biases
    # O, S, D -- pump parameters
    # sigma -- sqrt(std) for random number
    # alpha -- momentum parameter
    # c_th -- restriction on amplitudes growth
    def evolve(self):
        # choosing an attempt for the amplitude evolution 
        #spins_current=torch.cuda.FloatTensor(self.dim, self.attempt_num)
        random_attempt = random.randint(1, self.attempt_num-1)
        er_current=torch.ones((self.dim, self.attempt_num), dtype=torch.float32, device=torch.device('cuda'))

        # initializing current and error amplitudes
        c_current = 0.05*torch.randn_like(er_current)
        #print(np.shape(c_current.cpu()))
        #er_current=torch.ones((self.dim, self.attempt_num), dtype=torch.float32, device=torch.device('cuda'))


        # creating the array for evolving amplitudes from random attempt for x and error
        c_evol = torch.cuda.FloatTensor(dim, N, device = torch.device('cuda'))
       # er_evol= torch.cuda.FloatTensor(dim, N)
        c_evol[:, 0] = c_current[:,random_attempt]
       # er_evol[:, 0] = er_current[:,random_attempt]

        spins_current=torch.sign(c_current)
        spins_opt = torch.sign(c_current)
    
        c_opt=torch.zeros_like(c_current)
        #Checking parameters
        p=torch.zeros((self.attempt_num), dtype=torch.float32, device=torch.device('cuda'))
        tar=torch.zeros_like(p)


        dc=torch.zeros((self.dim, self.attempt_num), dtype=torch.float32, device=torch.device('cuda'))
        de=torch.zeros_like(dc)

        #define time array
        t=torch.zeros_like(p)
    
        #c_all=torch.zeros((dim, attempt_num, N), dtype=torch.float32)

        #initializing critical time points, optimal energy, optimal max_cut , target amplitude arrays
        t_c=torch.zeros_like(p)
        betha=torch.zeros_like(p)

        t1=0
        t=0
        #Initial values of energy, energy_opt, cut and cut_opt

        H = torch.einsum('ijk,is,js,ks->s',(self.J, spins_current,spins_current, spins_current))
        H_opt = torch.einsum('ijk,is,js,ks->s',(self.J, spins_current,spins_current, spins_current))
         #2178#
#         H_evol=torch.zeros(self.N, device=torch.device('cuda'))
#         H_evol[0]=H_opt[random_attempt]
        #Initial values of target amplitude, pump and betha
        tar[:] = self.alpha 
        p[:] = 1 + self.pumping 
        # initializing moving average of amplitudes increment
#         dc_momentum = torch.zeros_like(dc)
        for i in range(1,self.N):
            t=i
            #Calculation time for the beta
            checkt1=torch.Tensor.float(((i-t_c)<self.tau))
            
            t1 += checkt1 * self.dt
            t1 = checkt1 * t1

            #Amplitude and error increment
            dc = self.ampl_inc(p, c_current, er_current)
            de = self.error_inc(c_current, tar, er_current, betha)

            #Amplitude, error and spins on current iteration
          
            c_current += dc * self.dt
            er_current += de * self.dt
            mean_error = torch.mean(er_current + de * self.dt, axis = 0)
            check321 = torch.Tensor.float(mean_error < self.GGamma)
            er_current = check321 * er_current + (1-check321)*er_current/mean_error
            check123=torch.Tensor.float(er_current>=0)
            er_current=check123*er_current
            spins_current=torch.sign(c_current)
            
            #Additional tensors for full dynamics visualization
#             c_evol[:,i] = c_current[:,random_attempt]
            #er_evol[:,i] = er_current[:,random_attempt]
            #c_all[:,:,i]=c_current
            #spins_all = torch.sign(c_all)
            #spins=torch.sign(c_current)
            
            #Calculation of the energy and cut on the current time-step
            H = torch.einsum('ijk,is,js,ks->s',(self.J, spins_current,spins_current, spins_current))
            check=torch.Tensor.float(H<H_opt)
            H_opt=check*H+(1-check)*H_opt
            c_opt=check*c_current+(1-check)*c_opt

            tanh = self.tanh(H, H_opt)
            tar = self.alpha - tanh
            p = (1 + self.pumping * self.c_str)  + tanh
            betha = self.beta(t1)
#             spins_opt = torch.sign(c_opt)
            #Checking of the critical time
#             check=torch.Tensor.float(H<H_opt)
         
            #print(H.min(), H_opt.min())
            #Calculation of the critical time
            t_c=check*t+(1-check)*t_c
            
            checktime1=torch.Tensor.float((t-t_c)>=self.tau)
                #Checking critical time
            t_c=checktime1*t+(1-checktime1)*t_c
            
            #Calculation of the optimal energy and optimal cut
#             H_opt=check*H+(1-check)*H_opt
            if torch.any(torch.isnan(c_current)):
                break
              
            else:
                spins_opt=check * spins_current+(1-check) * spins_opt
                #Comparing time w/o cut increase

            
            
#             H_evol[i]=torch.einsum('ijk,is,js,ks->s',(self.J, spins_opt,spins_opt, spins_opt))[random_attempt]
        self.lastcuts = H_opt 

        return self.lastcuts, spins_opt

In [ ]:
class SimcimInterface(mli.Interface):
    def __init__(self, simcim, attempt_num, clip_percents=50): 
        super(SimcimInterface, self).__init__()
        self.simcim=simcim
        self.attemt_num = attempt_num
        self.simcim.evolve()
        self.perc = np.percentile(np.array(simcim.lastcuts.cpu()), clip_percents)
        self.scale = np.min(np.array(simcim.lastcuts.cpu()))#+self.perc
        self.minimum = np.min(np.array(simcim.lastcuts.cpu()))
        
        
    def get_sim_params(self, packed_params):
        num_single = 7
        logdt, logzeta, logrho, logalpha, logdelta, logpumping, loggamma  = packed_params[:num_single]
        
        dt = 10**logdt
        zeta=10**logzeta
        rho=10**logrho
        alpha=10**logalpha
        delta=10**logdelta
        pumping=10**logpumping
        gamma=10**loggamma
     


        return dt, zeta, rho, alpha, delta, pumping, gamma
        
        
    def get_next_cost_dict(self, params_dict):
        params = params_dict['params']
        
        dt, zeta, rho, alpha, delta, pumping, gamma = self.get_sim_params(params)
        
        print('\nSIMCIM:\ndt={}\nzeta={}\nrho={}\nalpha={}\ndelta={}\npumping={}\ngamma={}\n'.format(dt, zeta, rho, alpha, delta, pumping, gamma))
        
        self.simcim.dt = dt
        self.simcim.zeta=zeta
        self.simcim.rho=rho
        self.simcim.alpha=alpha
        self.simcim.delta=delta
        self.simcim.pumping=pumping
        self.simcim.gamma=gamma
        self.simcim.evolve()
        cost_perc = min([np.percentile(np.array(self.simcim.lastcuts.cpu()), clip_percents)/np.abs(self.perc),1]) #np.min(self.simcim.lastcuts)
        cost_min = min([np.min(np.array(self.simcim.lastcuts.cpu()))/np.abs(self.scale),1]) 
        cost=cost_min+0.25*cost_perc
        uncer = 0
        bad = False
        print('Initial perc', self.perc)
        print('Final perc', np.percentile(np.array(self.simcim.lastcuts.cpu()), clip_percents))
        print('Final energy', np.min(np.array(self.simcim.lastcuts.cpu())))
        cost_dict = {'cost':cost, 'uncer':uncer, 'bad':bad}
        return cost_dict

In [ ]:
for i in range(0, 10):
    psize=int(10*(i+1))
    for number in range(0, 10):
        print(i, psize, number)
        foldername = str(psize)+'_'+str(number)
        dt, zeta, rho, alpha, delta, pumping, gamma = np.load(DATA_ROOT+'/Sparse random/Leleu_each/'+foldername+'_params.npy')
        K= torch.tensor(np.load(DATA_ROOT+'/Sparse random/K_sparse_random_'+foldername+'_.npy'), device=torch.device('cuda'))
    #K = K_imported.reshape(100,100,100)
        unity=torch.ones((K.size(0)), dtype=torch.float32).cuda('cuda')
        dim=K.size(0)
        N = 1000
        attempt_num =2000
        dim = K.size(0)
        tau=K.size(0)
#         if number==0:
        attempt_num = 1000
        percentile = 50
        population_size=50
        clip_percents=50
        sim=SIMCIM(K, alpha,rho,delta,gamma,pumping, N,dt, dim,  zeta, attempt_num,  device='cuda')
        interface = SimcimInterface(sim,attempt_num, percentile)
        
        first_params = np.array([np.max([np.min([np.log10(sim.dt),3]),-3]), np.max([np.min([np.log10(sim.zeta),3]),-3]), np.max([np.min([np.log10(sim.rho),3]),-3]), np.max([np.min([np.log10(sim.alpha),3]),-3]), np.max([np.min([np.log10(sim.delta),3]),-3]), np.max([np.min([np.log10(sim.pumping),3]),-3]), np.max([np.min([np.log10(sim.gamma),3]),-3])])
        min_boundary = np.array([-3,  -3, -3,  -3, -3, -3, -3])
        max_boundary = np.array([-0., 3,  3,   3, 3, 3, 3])
        num_params = 7

        config = dict(controller_type='neural_net', 
                      training_type = 'differential_evolution',
                      first_params = first_params,
                      min_boundary = min_boundary, 
                      max_boundary = max_boundary,
                      num_params = num_params,
                      trust_region = 0.5,#0.5
                      max_num_runs = 1500,
                      num_training_runs = 2 * num_params,
                      log_level = 0, 
                      no_delay = False, 
                      max_num_runs_without_better_params = 150, 
                      #evolution_strategy='best1', 
                      population_size=10, 
                      mutation_scale = (0.5,1.0), 
                      cross_over_probability = 0.7, 
                      default_bad_cost = 1.25, 
                      #target_cost=-(8+np.percentile(np.array(sim.lastcuts.cpu()), clip_percents))/(-np.min(np.array(sim.lastcuts.cpu()))+np.percentile(np.array(sim.lastcuts.cpu()), clip_percents)),
                      default_bad_uncertainty = 0.1,
                      #predict_global_minima_at_end = True,
                      cost_has_noise = True)

        controller = mlc.create_controller(interface, **config)
        controller.optimize()

        clear_output(True)

        print('Best cut found:')
        print(controller.best_cost*controller.interface.scale + controller.interface.perc)

        mlv.show_all_default_visualizations(controller)
        dt=10**controller.best_params[0]
        zeta=10**controller.best_params[1]
        rho=10**controller.best_params[2]
        alpha=10**controller.best_params[3]
        delta=10**controller.best_params[4]
        pumping=10**controller.best_params[5]
        gamma=10**controller.best_params[6]
            #         delta=14.

        attempt_num =2000

        params=np.array([dt, zeta, rho, alpha, delta, pumping, gamma])
        
        sim=SIMCIM(K, alpha,rho,delta,gamma,pumping, N,dt, dim,  zeta, attempt_num,  device='cuda')
        H_opt, spins_opt =sim.evolve()
        spins=torch.sign(spins_opt).double()
        print(H_opt.min())
        K_doub=K.double()
        energy=torch.einsum('ijk,is,js,ks->s',(K_doub,spins,spins,spins))
#         while energy.min().item()==0:
#             print('error')
#             sim=SIMCIM(K, alpha,rho,delta,gamma,pumping, N,dt, dim,  zeta, attempt_num,  device='cuda')
#             H_opt, spins_opt, c_evol, H_evol=sim.evolve()
#             spins=torch.sign(spins_opt).double()
#             K_doub=K.double()
#             energy=torch.einsum('ijk,is,js,ks->s',(K_doub,spins,spins,spins))
#         for l in range(K.size(0)):
#             plt.plot(c_evol.cpu()[l,:])
#         plt.show()
#         plt.plot(H_evol.cpu())
#         plt.show()
        
        spins=torch.sign(spins_opt).double()
        K_doub=K.double()
        energy=torch.einsum('ijk,is,js,ks->s',(K_doub,spins,spins,spins))
        plt.hist(H_opt.cpu(), bins=40)
        plt.hist(energy.cpu(), bins=40)
        plt.show()
        num = np.argmin(energy.cpu())
        print(energy.cpu().min(), energy.cpu().max())
        print(H_opt.min(), H_opt.max())
        np.save(DATA_ROOT+'/Sparse random/Leleu_each/'+foldername+'_bit.npy', np.array(spins.cpu())[:, num])
        np.save(DATA_ROOT+'/Sparse random/Leleu_each/'+foldername+'.npy', np.array(H_opt.cpu(), dtype=np.float64))
        np.save(DATA_ROOT+'/Sparse random/Leleu_each/'+foldername+'_params.npy', params)

# Neuron + Differrential

In [ ]:
import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv